# 03 — Preprocessing (Phase 3)

Using the finalized feature audit from `02_feature_leakage_audit.ipynb`, this notebook builds the preprocessing machinery needed before modeling: feature selection, the `pdays` leakage-safe recoding, imputation/encoding/scaling pipelines, and reproducible train/validation/test splits.

**No model is trained and no hyperparameters are tuned here.** The only "fitting" that happens is for preprocessing transformers (imputers, encoder, scaler), and only on the training split — this is standard practice to prevent information from validation/test leaking into how missing values or categories are handled. The original dataset file is never modified.

In [4]:
import os
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

pd.set_option("display.max_columns", 100)

## Load the dataset (read-only)

In [5]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df_raw = pd.read_csv(DATA_PATH, sep=";")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 41188, Columns: 21


## 1. Select the approved primary features and target

Per the finalized audit (`results/feature_availability_audit.csv`), the 13 approved primary features are the customer-profile, previous-campaign-history, and scheduled-contact-plan columns. We select exactly these plus the target `y` from the raw dataframe, and work on this subset (`df_raw` itself is left untouched).

In [6]:
PRIMARY_FEATURES = [
    # customer profile
    "age", "job", "marital", "education", "default", "housing", "loan",
    # previous campaign history
    "pdays", "previous", "poutcome",
    # scheduled contact plan
    "contact", "month", "day_of_week",
]
TARGET = "y"

assert len(PRIMARY_FEATURES) == 13

data = df_raw[PRIMARY_FEATURES + [TARGET]].copy()
print(f"Selected shape: {data.shape}")
data.head()

Selected shape: (41188, 14)


,age,job,marital,education,default,housing,loan,pdays,previous,poutcome,contact,month,day_of_week,y
0,56,housemaid,married,basic.4y,no,no,no,999,0,nonexistent,telephone,may,mon,no
1,57,services,married,high.school,unknown,no,no,999,0,nonexistent,telephone,may,mon,no
2,37,services,married,high.school,no,yes,no,999,0,nonexistent,telephone,may,mon,no
3,40,admin.,married,basic.6y,no,no,no,999,0,nonexistent,telephone,may,mon,no
4,56,services,married,high.school,no,no,yes,999,0,nonexistent,telephone,may,mon,no


## 2. Handle `pdays = 999` (sentinel for "never previously contacted")

`pdays` encodes "never contacted" as the sentinel value `999`, not a true missing value — mixing it into a numeric imputer or scaler as-is would badly distort both. We split it into two honest columns:

- `previously_contacted` (binary): `1` if the client was contacted in a prior campaign (`pdays != 999`), else `0`.
- `days_since_previous_contact`: the real day-count where it's meaningful, and `NaN` (not `999`) where the client was never previously contacted — so downstream imputation treats it as genuinely missing rather than as a numeric outlier.

The original `pdays` column is then dropped from this working copy (`df_raw` is unaffected).

In [7]:
data["previously_contacted"] = (data["pdays"] != 999).astype(int)
data["days_since_previous_contact"] = data["pdays"].replace(999, np.nan)
data = data.drop(columns=["pdays"])

print("previously_contacted value counts:")
print(data["previously_contacted"].value_counts())
print("\ndays_since_previous_contact summary (NaN = never previously contacted):")
print(data["days_since_previous_contact"].describe())
print("\nNaN count:", data["days_since_previous_contact"].isna().sum())

data.head()

previously_contacted value counts:
previously_contacted
0    39673
1     1515
Name: count, dtype: int64

days_since_previous_contact summary (NaN = never previously contacted):
count    1515.000000
mean        6.014521
std         3.824906
min         0.000000
25%         3.000000
50%         6.000000
75%         7.000000
max        27.000000
Name: days_since_previous_contact, dtype: float64

NaN count: 39673


,age,job,marital,education,default,housing,loan,previous,poutcome,contact,month,day_of_week,y,previously_contacted,days_since_previous_contact
0,56,housemaid,married,basic.4y,no,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
1,57,services,married,high.school,unknown,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
2,37,services,married,high.school,no,yes,no,0,nonexistent,telephone,may,mon,no,0,NaN
3,40,admin.,married,basic.6y,no,no,no,0,nonexistent,telephone,may,mon,no,0,NaN
4,56,services,married,high.school,no,no,yes,0,nonexistent,telephone,may,mon,no,0,NaN


## 3. Preserve categorical `"unknown"` values as their own category

Several categorical columns (`job`, `marital`, `education`, `default`, `housing`, `loan`) use the literal string `"unknown"` for missing information. We deliberately **do not** treat `"unknown"` as a null to be imputed — it is itself informative (e.g. a client declining to disclose `default` status may differ systematically from one who answers `"no"`). It is left as-is and will simply pass through the categorical pipeline as its own one-hot category.

In [8]:
categorical_features = ["job", "marital", "education", "default", "housing", "loan",
                         "poutcome", "contact", "month", "day_of_week"]
numerical_features = ["age", "previous", "days_since_previous_contact", "previously_contacted"]

assert set(categorical_features) | set(numerical_features) == set(data.columns) - {TARGET}

unknown_counts = (data[categorical_features] == "unknown").sum()
print("'unknown' counts kept as a category (not imputed):")
print(unknown_counts[unknown_counts > 0])

'unknown' counts kept as a category (not imputed):
job           330
marital        80
education    1731
default      8597
housing       990
loan          990
dtype: int64


## 4. Numerical and categorical preprocessing pipelines

Two `ColumnTransformer` preprocessors are built, sharing the same imputation/encoding strategy but differing in scaling:

- **Numerical features** (`age`, `previous`, `days_since_previous_contact`, `previously_contacted`): median imputation for missing values (only `days_since_previous_contact` actually has any).
- **Categorical features**: most-frequent imputation (a no-op here since there are no true NaNs — `"unknown"` is a real category, not missing) followed by `OneHotEncoder(handle_unknown="ignore")`, so categories unseen at fit time don't crash inference.
- **Logistic Regression preprocessor**: adds `StandardScaler()` after numeric imputation, since LR is scale-sensitive.
- **Random Forest preprocessor**: no scaling on numeric features, since tree-based models are invariant to monotonic feature scaling.

Separate pipeline instances are built for each so that fitting one does not affect the other's internal state.

In [9]:
def make_categorical_pipeline():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])


# Random Forest: median impute numerics, no scaling.
preprocessor_rf = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), numerical_features),
    ("cat", make_categorical_pipeline(), categorical_features),
])

# Logistic Regression: median impute + standardize numerics.
preprocessor_lr = ColumnTransformer(transformers=[
    ("num", Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numerical_features),
    ("cat", make_categorical_pipeline(), categorical_features),
])

preprocessor_rf, preprocessor_lr

(ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                  ['age', 'previous',
                                   'days_since_previous_contact',
                                   'previously_contacted']),
                                 ('cat',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('onehot',
                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                  ['job', 'marital', 'education', 'default',
                                   'housing', 'loan', 'poutcome', 'contact',
                                   'month', 'day_of_week'])]),
 ColumnTransformer(transformers=[('num',
                                  Pipeline(steps=[('imputer',
                                                   SimpleImputer(strategy

## 5. Reproducible 70:15:15 splits

Two different split strategies are built, each reproducible via a fixed `random_state` (for the stratified split) and deterministic index slicing (for the chronological split):

- **Stratified random split**: rows are shuffled, with `y`'s class proportions preserved in train/validation/test. Good for an i.i.d.-style evaluation.
- **Chronological split**: the original row order is preserved as-is (no shuffling), with the first 70% of rows as train, the next 15% as validation, and the final 15% as test. This simulates deploying a model trained on past data to predict on data that comes later — closer to real-world use, and a useful check for whether performance degrades once economic conditions or campaign strategy shift over time.

Both splits operate on row indices only; no row is duplicated or dropped.

In [10]:
# --- Stratified random split ---
strat_train_idx, strat_temp_idx = train_test_split(
    data.index,
    test_size=0.30,
    stratify=data[TARGET],
    random_state=RANDOM_STATE,
)
strat_val_idx, strat_test_idx = train_test_split(
    strat_temp_idx,
    test_size=0.50,
    stratify=data.loc[strat_temp_idx, TARGET],
    random_state=RANDOM_STATE,
)

print("Stratified split sizes:", len(strat_train_idx), len(strat_val_idx), len(strat_test_idx))
print("Stratified split proportions of y='yes':")
for name, idx in [("train", strat_train_idx), ("val", strat_val_idx), ("test", strat_test_idx)]:
    print(f"  {name}: {(data.loc[idx, TARGET] == 'yes').mean():.4f}")

Stratified split sizes: 28831 6178 6179
Stratified split proportions of y='yes':
  train: 0.1127
  val: 0.1127
  test: 0.1126


In [11]:
# --- Chronological split (original row order preserved) ---
n = len(data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

chron_train_idx = data.index[:train_end]
chron_val_idx = data.index[train_end:val_end]
chron_test_idx = data.index[val_end:]

print("Chronological split sizes:", len(chron_train_idx), len(chron_val_idx), len(chron_test_idx))
print("Chronological split proportions of y='yes':")
for name, idx in [("train", chron_train_idx), ("val", chron_val_idx), ("test", chron_test_idx)]:
    print(f"  {name}: {(data.loc[idx, TARGET] == 'yes').mean():.4f}")

Chronological split sizes: 28831 6178 6179
Chronological split proportions of y='yes':
  train: 0.0557
  val: 0.1065
  test: 0.3845


## 6. Fit preprocessing on the training split only

We use the **stratified split** as the primary split for fitting/inspecting the preprocessors (the chronological split's indices are saved for later use in modeling, to compare i.i.d. vs. time-respecting evaluation). Both `preprocessor_rf` and `preprocessor_lr` are `fit_transform`-ed on the training rows only; validation and test rows are only ever `.transform()`-ed, never seen during `fit`. This means imputation values (medians/modes), one-hot category vocabularies, and scaler means/variances all come exclusively from the training data.

In [12]:
X_train = data.loc[strat_train_idx, numerical_features + categorical_features]
X_val = data.loc[strat_val_idx, numerical_features + categorical_features]
X_test = data.loc[strat_test_idx, numerical_features + categorical_features]

y_train = data.loc[strat_train_idx, TARGET]
y_val = data.loc[strat_val_idx, TARGET]
y_test = data.loc[strat_test_idx, TARGET]

# Fit ONLY on training data; transform (never fit) validation/test.
X_train_rf = preprocessor_rf.fit_transform(X_train)
X_val_rf = preprocessor_rf.transform(X_val)
X_test_rf = preprocessor_rf.transform(X_test)

X_train_lr = preprocessor_lr.fit_transform(X_train)
X_val_lr = preprocessor_lr.transform(X_val)
X_test_lr = preprocessor_lr.transform(X_test)

print("RF preprocessor  — train/val/test shapes:", X_train_rf.shape, X_val_rf.shape, X_test_rf.shape)
print("LR preprocessor  — train/val/test shapes:", X_train_lr.shape, X_val_lr.shape, X_test_lr.shape)

RF preprocessor  — train/val/test shapes: (28831, 57) (6178, 57) (6179, 57)
LR preprocessor  — train/val/test shapes: (28831, 57) (6178, 57) (6179, 57)


## 7. Inspect transformed dimensions and verify unseen-category handling

First we list the output feature names (confirming the one-hot expansion accounts for the extra columns), then we deliberately construct a row with a category value that does not exist anywhere in the training data, and confirm the fitted preprocessor transforms it without error — `OneHotEncoder(handle_unknown="ignore")` should encode it as all-zeros for that feature's one-hot block, rather than raising.

In [13]:
feature_names_rf = preprocessor_rf.get_feature_names_out()
feature_names_lr = preprocessor_lr.get_feature_names_out()

print(f"RF preprocessor output: {len(feature_names_rf)} columns "
      f"({len(numerical_features)} numeric + {len(feature_names_rf) - len(numerical_features)} one-hot)")
print(f"LR preprocessor output: {len(feature_names_lr)} columns "
      f"({len(numerical_features)} numeric + {len(feature_names_lr) - len(numerical_features)} one-hot)")

assert list(feature_names_rf) == list(feature_names_lr), "RF/LR encoders should produce identical columns."
feature_names_rf[:20]

RF preprocessor output: 57 columns (4 numeric + 53 one-hot)
LR preprocessor output: 57 columns (4 numeric + 53 one-hot)


array(['num__age', 'num__previous', 'num__days_since_previous_contact',
       'num__previously_contacted', 'cat__job_admin.',
       'cat__job_blue-collar', 'cat__job_entrepreneur',
       'cat__job_housemaid', 'cat__job_management', 'cat__job_retired',
       'cat__job_self-employed', 'cat__job_services', 'cat__job_student',
       'cat__job_technician', 'cat__job_unemployed', 'cat__job_unknown',
       'cat__marital_divorced', 'cat__marital_married',
       'cat__marital_single', 'cat__marital_unknown'], dtype=object)

In [14]:
# Construct a synthetic row with a category never seen in training, for a column
# fit on the training split (e.g. an impossible 'job' value).
synthetic_row = X_train.iloc[[0]].copy()
synthetic_row["job"] = "this_category_was_never_seen_in_training"

assert "this_category_was_never_seen_in_training" not in X_train["job"].unique()

try:
    encoded_rf = preprocessor_rf.transform(synthetic_row)
    encoded_lr = preprocessor_lr.transform(synthetic_row)
    print("Unseen category handled without error (handle_unknown='ignore').")
except ValueError as e:
    print("FAILED — unseen category raised an error:", e)

encoded_rf_dense = encoded_rf.toarray() if hasattr(encoded_rf, "toarray") else np.asarray(encoded_rf)
job_cols = [i for i, name in enumerate(feature_names_rf) if name.startswith("cat__job_")]
print("One-hot block for 'job' on the unseen-category row (all zeros expected):",
      encoded_rf_dense[0, job_cols])

Unseen category handled without error (handle_unknown='ignore').
One-hot block for 'job' on the unseen-category row (all zeros expected): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [15]:
# Also check whether any *real* categories in val/test are actually absent from train
# (for this dataset's low-cardinality columns this is expected to be empty, but the
# check — and the pipeline's robustness to it — matters in general).
unseen_report = {}
for col in categorical_features:
    train_categories = set(X_train[col].unique())
    for split_name, split_X in [("val", X_val), ("test", X_test)]:
        unseen = set(split_X[col].unique()) - train_categories
        if unseen:
            unseen_report[(col, split_name)] = unseen

if unseen_report:
    print("Real unseen categories found (would be ignored safely by the fitted encoder):")
    for (col, split_name), vals in unseen_report.items():
        print(f"  {col} ({split_name}): {vals}")
else:
    print("No real unseen categories in val/test for this dataset — "
          "robustness was confirmed above via the synthetic row instead.")

No real unseen categories in val/test for this dataset — robustness was confirmed above via the synthetic row instead.


## 8. Save split indices and preprocessing summaries

We persist:

- `results/split_stratified.csv` and `results/split_chronological.csv` — one row per original dataset index, labeled with its split (`train`/`val`/`test`) and strategy, so any later notebook can reproduce the exact same rows without re-deriving the split.
- `results/preprocessing_summary.json` — a machine-readable record of the feature lists, the `pdays` recoding, the pipeline configuration, and the resulting transformed dimensions, for traceability.

No fitted transformer objects are pickled here — only metadata — since no modeling happens in this notebook.

In [16]:
def build_split_table(train_idx, val_idx, test_idx):
    frames = [
        pd.DataFrame({"row_index": train_idx, "split": "train"}),
        pd.DataFrame({"row_index": val_idx, "split": "val"}),
        pd.DataFrame({"row_index": test_idx, "split": "test"}),
    ]
    return pd.concat(frames, ignore_index=True).sort_values("row_index").reset_index(drop=True)


strat_split_table = build_split_table(strat_train_idx, strat_val_idx, strat_test_idx)
chron_split_table = build_split_table(chron_train_idx, chron_val_idx, chron_test_idx)

strat_split_path = f"{RESULTS_DIR}/split_stratified.csv"
chron_split_path = f"{RESULTS_DIR}/split_chronological.csv"
strat_split_table.to_csv(strat_split_path, index=False)
chron_split_table.to_csv(chron_split_path, index=False)

print("Saved:", strat_split_path, "-", strat_split_table.shape)
print("Saved:", chron_split_path, "-", chron_split_table.shape)

Saved: ../results/split_stratified.csv - (41188, 2)
Saved: ../results/split_chronological.csv - (41188, 2)


In [17]:
preprocessing_summary = {
    "random_state": RANDOM_STATE,
    "primary_features_selected": PRIMARY_FEATURES,
    "target": TARGET,
    "pdays_recoding": {
        "sentinel_value": 999,
        "new_columns": ["previously_contacted", "days_since_previous_contact"],
        "dropped_column": "pdays",
        "previously_contacted_counts": {str(k): int(v) for k, v in data["previously_contacted"].value_counts().items()},
        "days_since_previous_contact_nan_count": int(data["days_since_previous_contact"].isna().sum()),
    },
    "final_feature_columns": {
        "numerical": numerical_features,
        "categorical": categorical_features,
        "total_raw_columns": len(numerical_features) + len(categorical_features),
    },
    "unknown_category_counts": {str(k): int(v) for k, v in unknown_counts[unknown_counts > 0].items()},
    "pipelines": {
        "random_forest": {
            "numerical": "SimpleImputer(strategy='median')",
            "categorical": "SimpleImputer(strategy='most_frequent') -> OneHotEncoder(handle_unknown='ignore')",
            "scaling": "none",
        },
        "logistic_regression": {
            "numerical": "SimpleImputer(strategy='median') -> StandardScaler()",
            "categorical": "SimpleImputer(strategy='most_frequent') -> OneHotEncoder(handle_unknown='ignore')",
            "scaling": "StandardScaler on numerical features",
        },
    },
    "transformed_dimensions": {
        "n_output_columns": int(len(feature_names_rf)),
        "n_numerical": len(numerical_features),
        "n_onehot": int(len(feature_names_rf) - len(numerical_features)),
    },
    "unseen_category_handling_verified": True,
    "splits": {
        "stratified": {
            "strategy": "random, stratified on y, random_state fixed",
            "train_size": len(strat_train_idx),
            "val_size": len(strat_val_idx),
            "test_size": len(strat_test_idx),
        },
        "chronological": {
            "strategy": "original row order preserved, first 70% / next 15% / last 15%",
            "train_size": len(chron_train_idx),
            "val_size": len(chron_val_idx),
            "test_size": len(chron_test_idx),
        },
    },
    "fit_on_training_only": True,
}

summary_path = f"{RESULTS_DIR}/preprocessing_summary.json"
with open(summary_path, "w") as f:
    json.dump(preprocessing_summary, f, indent=2)

print("Saved:", summary_path)
preprocessing_summary

Saved: ../results/preprocessing_summary.json


{'random_state': 42,
 'primary_features_selected': ['age',
  'job',
  'marital',
  'education',
  'default',
  'housing',
  'loan',
  'pdays',
  'previous',
  'poutcome',
  'contact',
  'month',
  'day_of_week'],
 'target': 'y',
 'pdays_recoding': {'sentinel_value': 999,
  'new_columns': ['previously_contacted', 'days_since_previous_contact'],
  'dropped_column': 'pdays',
  'previously_contacted_counts': {'0': 39673, '1': 1515},
  'days_since_previous_contact_nan_count': 39673},
 'final_feature_columns': {'numerical': ['age',
   'previous',
   'days_since_previous_contact',
   'previously_contacted'],
  'categorical': ['job',
   'marital',
   'education',
   'default',
   'housing',
   'loan',
   'poutcome',
   'contact',
   'month',
   'day_of_week'],
  'total_raw_columns': 14},
 'unknown_category_counts': {'job': 330,
  'marital': 80,
  'education': 1731,
  'default': 8597,
  'housing': 990,
  'loan': 990},
 'pipelines': {'random_forest': {'numerical': "SimpleImputer(strategy='median

## Summary

- Selected the 13 approved primary features + target; `pdays` was recoded into `previously_contacted` and `days_since_previous_contact` (NaN for "never contacted"), leaving 14 raw working columns (4 numerical, 10 categorical).
- `"unknown"` categorical values were preserved as a real category, not imputed.
- Two `ColumnTransformer` preprocessors were built — identical imputation/encoding, differing only in that the Logistic Regression one standardizes numeric features and the Random Forest one does not.
- Preprocessors were fit **only** on the stratified training split; validation/test were only transformed.
- Verified the one-hot encoder safely handles a category never seen during fit (`handle_unknown="ignore"`), with no real unseen categories present in this dataset's val/test splits.
- Saved `results/split_stratified.csv`, `results/split_chronological.csv`, and `results/preprocessing_summary.json`.

No model was trained, no hyperparameters were tuned, and the original dataset file was not modified.